# 03 — Quantile Forecast Baseline

This notebook demonstrates the end-to-end quantile forecasting pipeline:

1. **Load Gold price features** from S3 (`features/price_features/`)
2. **Train custom Quantile GBM** (from scratch, no sklearn/LightGBM)
3. **Evaluate** pinball loss and prediction intervals
4. **Visualize** forecasts with uncertainty bands

**Forecast horizons:** 7d, 15d, 30d

**Model:** Custom Quantile Gradient-Boosted Trees with pinball loss

In [ ]:
import io
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name != "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

from src.models.quantile_gbm.gradient_boosted_trees import QuantileGBM
from src.models.quantile_gbm.loss import pinball_loss

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True

## 1. Load Gold Price Features from S3

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "aws_config.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

session = boto3.Session(
    profile_name=cfg["aws"].get("profile"),
    region_name=cfg["aws"].get("region", "ap-south-1"),
)
s3 = session.client("s3")
bucket = cfg["s3"]["bucket"]

# List and read all price feature parquet files
prefix = "features/price_features/"
keys, token = [], None
while True:
    kwargs = dict(Bucket=bucket, Prefix=prefix)
    if token:
        kwargs["ContinuationToken"] = token
    resp = s3.list_objects_v2(**kwargs)
    keys += [o["Key"] for o in resp.get("Contents", []) if o["Key"].endswith(".parquet")]
    if resp.get("IsTruncated"):
        token = resp["NextContinuationToken"]
    else:
        break

print(f"Found {len(keys)} parquet files under s3://{bucket}/{prefix}")

import re
frames = []
for key in keys:
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    table = pq.read_table(buf)
    df_part = table.to_pandas()
    for m in re.finditer(r"([^/=]+)=([^/]+)/", key):
        col, val = m.group(1), m.group(2)
        if col not in df_part.columns:
            df_part[col] = val
    frames.append(df_part)

df = pd.concat(frames, ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
df.head()

## 2. Select Commodity & Prepare Features

In [ ]:
# Find top commodities by data volume
commodity_counts = df.groupby("commodity")["modal_price"].count().sort_values(ascending=False)
print("Top 10 commodities by row count:")
print(commodity_counts.head(10).to_string())

TOP_COMMODITY = commodity_counts.index[0]
print(f"\nSelected commodity: {TOP_COMMODITY}")

In [ ]:
# Feature columns (must match build_price_features.py output)
FEATURE_COLS = [
    "price_lag_1d", "price_lag_7d", "price_lag_14d", "price_lag_30d",
    "price_mean_7d", "price_std_7d",
    "price_mean_14d", "price_std_14d",
    "price_mean_30d", "price_std_30d",
    "price_momentum_7d",
    "arrivals_tonnes", "arrivals_mean_7d",
    "temp_mean_7d", "precip_sum_7d", "humidity_mean_7d",
    "ndvi", "ndvi_delta_30d",
    "food_cpi_index", "food_wpi_index",
    "day_of_week", "day_of_month", "month", "is_weekend",
]
HORIZONS = [7, 15, 30]
QUANTILES = [0.10, 0.50, 0.90]

## 3. Train/Test Split & Model Training

In [ ]:
commodity_df = df[df["commodity"] == TOP_COMMODITY].sort_values("date").copy()
feat_cols = [c for c in FEATURE_COLS if c in commodity_df.columns]
print(f"Commodity rows: {len(commodity_df):,}, features available: {len(feat_cols)}")

results = {}

for horizon in HORIZONS:
    target_col = f"target_price_{horizon}d"
    if target_col not in commodity_df.columns:
        print(f"  Skipping {horizon}d — target column missing")
        continue

    sub = commodity_df.dropna(subset=[target_col] + feat_cols).copy()
    if len(sub) < 200:
        print(f"  Skipping {horizon}d — only {len(sub)} rows (need ≥200)")
        continue

    # Time-based split: 70% train, 15% val, 15% test
    n = len(sub)
    train_cut = sub["date"].iloc[int(n * 0.70)]
    val_cut = sub["date"].iloc[int(n * 0.85)]

    train = sub[sub["date"] <= train_cut]
    val = sub[(sub["date"] > train_cut) & (sub["date"] <= val_cut)]
    test = sub[sub["date"] > val_cut]

    X_tr = train[feat_cols].values.astype(np.float64)
    y_tr = train[target_col].values.astype(np.float64)
    X_vl = val[feat_cols].values.astype(np.float64)
    y_vl = val[target_col].values.astype(np.float64)
    X_te = test[feat_cols].values.astype(np.float64)
    y_te = test[target_col].values.astype(np.float64)

    print(f"\n--- {TOP_COMMODITY} @ {horizon}d ---")
    print(f"  train={len(X_tr):,}  val={len(X_vl):,}  test={len(X_te):,}")

    # Train
    model = QuantileGBM(
        quantiles=QUANTILES,
        n_estimators=150,
        learning_rate=0.05,
        max_depth=4,
        min_samples_leaf=10,
        subsample=0.8,
        random_state=42,
    )
    model.fit(X_tr, y_tr, X_val=X_vl, y_val=y_vl)

    # Evaluate on test
    metrics = {"horizon": f"{horizon}d"}
    for q in QUANTILES:
        preds = model.predict_quantile(X_te, q)
        metrics[f"pinball_q{int(q*100):02d}"] = pinball_loss(y_te, preds, q)

    p50 = model.predict_quantile(X_te, 0.50)
    metrics["rmse"] = float(np.sqrt(np.mean((y_te - p50) ** 2)))
    metrics["mape"] = float(np.mean(np.abs((y_te - p50) / np.clip(np.abs(y_te), 1e-6, None))) * 100)
    results[horizon] = {
        "model": model, "metrics": metrics,
        "test_df": test.copy(), "y_te": y_te, "X_te": X_te,
    }
    print(f"  RMSE={metrics['rmse']:.2f}  MAPE={metrics['mape']:.1f}%")
    print(f"  Pinball q10={metrics['pinball_q10']:.4f}  q50={metrics['pinball_q50']:.4f}  q90={metrics['pinball_q90']:.4f}")

## 4. Metrics Summary

In [ ]:
if results:
    summary = pd.DataFrame([r["metrics"] for r in results.values()])
    print(summary.to_string(index=False))

## 5. Forecast Visualization — Prediction Intervals

In [ ]:
for horizon, res in results.items():
    model = res["model"]
    test_df = res["test_df"].sort_values("date")
    X_te = res["X_te"]
    y_te = res["y_te"]

    q10 = model.predict_quantile(X_te, 0.10)
    q50 = model.predict_quantile(X_te, 0.50)
    q90 = model.predict_quantile(X_te, 0.90)

    fig, ax = plt.subplots(figsize=(14, 5))
    dates = test_df["date"].values

    # Prediction interval band
    ax.fill_between(dates, q10, q90, alpha=0.25, color="steelblue", label="80% PI (q10–q90)")
    ax.plot(dates, q50, color="steelblue", linewidth=1.5, label="Median forecast (q50)")
    ax.plot(dates, y_te, color="black", linewidth=1, alpha=0.7, label="Actual")

    ax.set_title(f"{TOP_COMMODITY} — {horizon}d Forecast with Uncertainty")
    ax.set_xlabel("Date")
    ax.set_ylabel("Price")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Training Loss Curves

In [ ]:
for horizon, res in results.items():
    model = res["model"]
    losses = model.training_losses()

    fig, ax = plt.subplots(figsize=(10, 4))
    for q, loss_curve in losses.items():
        ax.plot(loss_curve, label=f"q={q:.2f}")
    ax.set_title(f"{TOP_COMMODITY} — {horizon}d Training Pinball Loss")
    ax.set_xlabel("Boosting Round")
    ax.set_ylabel("Mean Pinball Loss")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 7. Coverage Analysis

Check if the 80% prediction interval (q10–q90) actually captures ~80% of test observations.

In [ ]:
for horizon, res in results.items():
    model = res["model"]
    y_te = res["y_te"]
    X_te = res["X_te"]

    q10 = model.predict_quantile(X_te, 0.10)
    q90 = model.predict_quantile(X_te, 0.90)
    q50 = model.predict_quantile(X_te, 0.50)

    in_80 = np.mean((y_te >= q10) & (y_te <= q90)) * 100
    in_50 = np.mean((y_te >= q50) * 1.0) * 100  # q50 is median

    print(f"{horizon}d — 80% PI coverage: {in_80:.1f}% (target: 80%)")

## 8. Summary

**Key takeaways:**
- Custom Quantile GBM trains from scratch (no sklearn/LightGBM dependency)
- Pinball loss drives quantile-specific tree fitting
- Prediction intervals provide uncertainty quantification for lending decisions
- Coverage analysis validates calibration of prediction bands